In [31]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver

llm = init_chat_model("openai:gpt-4o-mini")

conn = sqlite3.connect(
    "memory.db", 
    check_same_thread=False,
)

In [32]:
class State(MessagesState):
    pass

graph_builder = StateGraph(State)

In [33]:
@tool
def get_weather(city: str):
    """Gets weather in City"""
    return f"The weather in {city} is sunny"

llm_with_tools = llm.bind_tools([get_weather])

def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [34]:
tool_node = ToolNode(
    tools=[get_weather]
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn)
)

In [36]:
graph.invoke(  
    {
        "messages": [
            {
                "role": "user",
                "content": "what is the weather in korea"
            }
        ]
    },
    config={
        "configurable": {
            "thread_id": "1"
        }
    }
)

{'messages': [HumanMessage(content='what is the weather in korea', additional_kwargs={}, response_metadata={}, id='5ef96eef-536e-42fd-84d0-a57cc108906f'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_HlHW9DLxzQ4GZR9iQXt0SGSI', 'function': {'arguments': '{"city":"Korea"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 48, 'total_tokens': 63, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_7348b557b7', 'id': 'chatcmpl-CfoWacCGywJKyOvjf96SYi75TkGPz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--d6b67e60-3136-4dbb-af13-1317a3359b04-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Korea'}, 'id': 'call_H

In [37]:
graph.invoke(  
    {
        "messages": [
            {
                "role": "user",
                "content": "what did I ask you?"
            }
        ]
    },
    config={
        "configurable": {
            "thread_id": "1"
        }
    }
)

{'messages': [HumanMessage(content='what is the weather in korea', additional_kwargs={}, response_metadata={}, id='5ef96eef-536e-42fd-84d0-a57cc108906f'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_HlHW9DLxzQ4GZR9iQXt0SGSI', 'function': {'arguments': '{"city":"Korea"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 48, 'total_tokens': 63, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_7348b557b7', 'id': 'chatcmpl-CfoWacCGywJKyOvjf96SYi75TkGPz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--d6b67e60-3136-4dbb-af13-1317a3359b04-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Korea'}, 'id': 'call_H